# Analisis Sentimen Komentar YouTube Karhutla 2026
**Alur:** muat data → validasi silang (TF-IDF vs IndoBERT) → pilih model dengan aturan yang ditetapkan sebelum melihat hasil → prediksi ke seluruh komentar → ringkasan.

**Persiapan:** Runtime > Change runtime type > **T4 GPU**. Siapkan dua file: `labeled_all.csv` dan `comments_clean.csv` (akan diminta unggah di sel 2).

**Catatan validitas (baca sebelum menulis kesimpulan):**
- Angka headline distribusi sentimen diambil dari **sampel acak berlabel manual** (dengan interval kepercayaan), bukan dari prediksi model.
- Prediksi model dipakai untuk **rincian** (per bulan, per video, per sasaran) dan dilaporkan sebagai estimasi, bukan fakta.

In [ ]:
RUN_BERT = True                      # False = hanya TF-IDF (tanpa GPU)
BERT_MODEL = "indolem/indobertweet-base-uncased"
BERT_FALLBACK = "indobenchmark/indobert-base-p1"
EPOCHS, LR, BATCH, MAXLEN = 4, 3e-5, 16, 128
SEED = 0
N_BOOT = 1000
%pip install -q transformers

In [ ]:
import os, math, random, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
if not (os.path.exists("labeled_all.csv") and os.path.exists("comments_clean.csv")):
    from google.colab import files
    print("Unggah labeled_all.csv dan comments_clean.csv"); files.upload()
L = pd.read_csv("labeled_all.csv"); C = pd.read_csv("comments_clean.csv")
print("berlabel:", len(L), "| seluruh komentar bersih:", len(C))
print(L.sent.value_counts().to_dict())

## 1. Fungsi bantu (model klasik, metrik, bootstrap)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

def tfidf_lr():
    feats = make_union(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                       TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True))
    return make_pipeline(feats, LogisticRegression(C=1, class_weight="balanced", max_iter=3000))

def folds(y, seed=SEED):
    return list(StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))

def oof_tfidf(X, y, make=tfidf_lr, classes=None):
    P = np.zeros((len(y), len(classes)))
    for tr, te in folds(y):
        m = make().fit(X[tr], y[tr]); P[te] = m.predict_proba(X[te])[:, [list(m.classes_).index(c) for c in classes]]
    return P

def oof_majority(y, classes):
    pred = np.empty(len(y), dtype=object)
    for tr, te in folds(y):
        pred[te] = pd.Series(y[tr]).value_counts().idxmax()
    return pred

def boot_ci(y, pred, n=N_BOOT, seed=1):
    rng = np.random.RandomState(seed); v = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y)); v.append(f1_score(y[i], pred[i], average="macro"))
    return np.percentile(v, [2.5, 97.5])

def paired_diff_ci(y, pa, pb, n=N_BOOT, seed=2):
    rng = np.random.RandomState(seed); d = []
    for _ in range(n):
        i = rng.randint(0, len(y), len(y)); d.append(f1_score(y[i], pa[i], average="macro") - f1_score(y[i], pb[i], average="macro"))
    return np.percentile(d, [2.5, 97.5])

def ringkas(nama, y, pred):
    lo, hi = boot_ci(y, pred)
    return {"model": nama, "macro_F1": round(f1_score(y, pred, average="macro"), 3), "CI95": f"[{lo:.3f}, {hi:.3f}]", "akurasi": round(accuracy_score(y, pred), 3)}

## 2. Kode IndoBERT (loop PyTorch sederhana, tanpa Trainer agar tahan perubahan versi)

In [ ]:
if RUN_BERT:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", DEVICE, "(sebaiknya cuda)")
    KEYS = ("input_ids", "attention_mask", "token_type_ids")
    _used = {"name": None}

    def _load():
        for name in (BERT_MODEL, BERT_FALLBACK):
            try:
                tok = AutoTokenizer.from_pretrained(name)
                mdl = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(_load.classes))
                _used["name"] = name; return tok, mdl
            except Exception as e:
                print("gagal memuat", name, "->", str(e)[:120])
        raise RuntimeError("Tidak ada model yang bisa dimuat")

    def bert_train_predict(texts_tr, y_tr, texts_te_list, classes, seed=SEED):
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        _load.classes = classes
        tok, model = _load(); model.to(DEVICE)
        cid = {c: i for i, c in enumerate(classes)}; yi = np.array([cid[v] for v in y_tr])
        cnt = np.bincount(yi, minlength=len(classes))
        w = torch.tensor(np.minimum(len(yi) / (len(classes) * np.maximum(cnt, 1)), 5.0), dtype=torch.float).to(DEVICE)
        enc = tok(list(texts_tr), truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
        n = len(yi); steps = EPOCHS * math.ceil(n / BATCH)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        lossf = torch.nn.CrossEntropyLoss(weight=w); model.train()
        for ep in range(EPOCHS):
            perm = np.random.permutation(n); tot = 0
            for i in range(0, n, BATCH):
                b = torch.as_tensor(perm[i:i+BATCH])
                batch = {k: enc[k][b].to(DEVICE) for k in KEYS if k in enc}
                loss = lossf(model(**batch).logits, torch.as_tensor(yi[b.numpy()]).to(DEVICE))
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad(); tot += loss.item() * len(b)
            print(f"   epoch {ep+1}/{EPOCHS} loss={tot/n:.4f}")
        model.eval(); outs = []
        for texts in texts_te_list:
            P = []
            with torch.no_grad():
                for i in range(0, len(texts), 64):
                    e = tok(list(texts[i:i+64]), truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
                    e = {k: e[k].to(DEVICE) for k in KEYS if k in e}
                    P.append(torch.softmax(model(**e).logits.float(), dim=-1).cpu().numpy())
            outs.append(np.vstack(P))
        del model, opt; torch.cuda.empty_cache(); return outs

    def oof_bert(Xraw, y, classes):
        P = np.zeros((len(y), len(classes)))
        for k, (tr, te) in enumerate(folds(y), 1):
            print(f"  fold {k}/5"); P[te] = bert_train_predict(Xraw[tr], y[tr], [Xraw[te]], classes)[0]
        return P

## 3. Validasi silang 5-fold untuk dua tugas: **sentimen** dan **sasaran (target)**
Aturan pemilihan model (ditetapkan sebelum melihat hasil): pilih model dengan macro-F1 tertinggi; jika selisih terhadap TF-IDF+LR tidak signifikan (CI 95% selisih mencakup 0), pakai TF-IDF+LR karena lebih sederhana dan stabil.

In [ ]:
TASKS = {"sent": L[L.ikut_model_sent].reset_index(drop=True), "target": L.reset_index(drop=True)}
CLASSES = {"sent": ["Negatif", "Netral", "Positif"], "target": sorted(L.target.unique())}
RES, OOF, CHOICE, SUMMARY = {}, {}, {}, []
for task, D in TASKS.items():
    classes = CLASSES[task]; y = D[task].values; Xc = D.text_clean.values; Xr = D.text.fillna("").values
    print(f"\n===== TUGAS: {task} | n={len(D)} | kelas: {pd.Series(y).value_counts().to_dict()}")
    preds = {"Mayoritas (pembanding)": oof_majority(y, classes)}
    P_lr = oof_tfidf(Xc, y, classes=classes); preds["TF-IDF + LR"] = np.array(classes, dtype=object)[P_lr.argmax(1)]; OOF[(task, "TF-IDF + LR")] = P_lr
    if RUN_BERT:
        P_b = oof_bert(Xr, y, classes); preds["IndoBERT"] = np.array(classes, dtype=object)[P_b.argmax(1)]; OOF[(task, "IndoBERT")] = P_b
    tab = pd.DataFrame([ringkas(n, y, p) for n, p in preds.items()]); print(tab.to_string(index=False)); SUMMARY.append(tab.assign(tugas=task))
    choice = "TF-IDF + LR"
    if RUN_BERT:
        lo, hi = paired_diff_ci(y, preds["IndoBERT"], preds["TF-IDF + LR"])
        print(f"Selisih macro-F1 IndoBERT - TF-IDF: CI95 [{lo:+.3f}, {hi:+.3f}]")
        f_b, f_l = f1_score(y, preds["IndoBERT"], average="macro"), f1_score(y, preds["TF-IDF + LR"], average="macro")
        if f_b > f_l and lo > 0: choice = "IndoBERT"
    CHOICE[task] = choice; RES[task] = preds; print("MODEL TERPILIH:", choice)
    print(classification_report(y, preds[choice], digits=3))

## 4. Cek bias: prediksi vs hitungan manual pada sampel acak

In [ ]:
D = TASKS["sent"]; m = (D.sumber == "acak").values; y = D.sent.values
man = pd.Series(y[m]).value_counts(normalize=True)
pr = pd.Series(RES["sent"][CHOICE["sent"]][m]).value_counts(normalize=True)
cek = pd.DataFrame({"manual_%": (man*100).round(1), "prediksi_OOF_%": (pr*100).round(1)}).fillna(0)
cek["selisih_poin"] = (cek["prediksi_OOF_%"] - cek["manual_%"]).round(1); print(cek)
print("\nJika selisih besar, headline pakai angka MANUAL; prediksi hanya untuk rincian relatif.")

## 5. Model akhir → prediksi ke seluruh komentar

In [ ]:
A = C.copy(); A["text"] = A.text.fillna("")
for task, D in TASKS.items():
    classes = CLASSES[task]; y = D[task].values; ch = CHOICE[task]; print(f"\n[{task}] melatih model akhir: {ch}")
    if ch == "IndoBERT":
        P = bert_train_predict(D.text.fillna("").values, y, [A.text.values], classes)[0]
    else:
        mdl = tfidf_lr().fit(D.text_clean.values, y); P = mdl.predict_proba(A.text_clean.values)[:, [list(mdl.classes_).index(c) for c in classes]]
    A["pred_" + task] = np.array(classes, dtype=object)[P.argmax(1)]
    for j, c in enumerate(classes): A[f"p_{task}_{c}"] = P[:, j].round(4)
lab_s = dict(zip(L.id, L.sent)); lab_t = dict(zip(L.id, L.target))
A["sent_final"] = [lab_s.get(i, p) for i, p in zip(A.comment_id, A.pred_sent)]
A["sent_final"] = A.sent_final.where(A.sent_final != "Tidak relevan")
A["target_final"] = [lab_t.get(i, p) for i, p in zip(A.comment_id, A.pred_target)]
A["sumber_label"] = np.where(A.comment_id.isin(L.id), "manual", "prediksi")
print(A.sumber_label.value_counts().to_dict())

## 6. Ringkasan untuk postingan

In [ ]:
from math import sqrt
def wilson(k, n, z=1.96):
    p = k/n; d = 1+z*z/n; c = (p+z*z/(2*n))/d; h = z*sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h, c+h
os.makedirs("hasil", exist_ok=True)
R = L[L.sumber == "acak"]; n = len(R); rows = []
for k, v in R.sent.value_counts().items():
    lo, hi = wilson(v, n); rows.append({"sentimen": k, "n": v, "persen": round(v/n*100, 1), "CI95": f"[{lo*100:.1f} - {hi*100:.1f}]"})
H = pd.DataFrame(rows); print(f"HEADLINE (sampel acak berlabel manual, n={n}):"); print(H.to_string(index=False)); H.to_csv("hasil/headline_manual.csv", index=False)

S = A.dropna(subset=["sent_final"]); print(f"\nESTIMASI MODEL pada {len(S)} komentar relevan:"); 
tot = (S.sent_final.value_counts(normalize=True)*100).round(1).rename("persen").to_frame(); print(tot); tot.to_csv("hasil/estimasi_model_total.csv")
S = S.assign(bulan=pd.to_datetime(S.published_at, utc=True).dt.strftime("%Y-%m"))
by_month = pd.crosstab(S.bulan, S.sent_final); by_month["n"] = by_month.sum(axis=1)
for c in ["Negatif","Netral","Positif"]: by_month[c+"_%"] = (by_month[c]/by_month.n*100).round(1)
print("\nPER BULAN:"); print(by_month); by_month.to_csv("hasil/per_bulan.csv")
by_t = pd.crosstab(S.target_final, S.sent_final); by_t["n"] = by_t.sum(axis=1); by_t["share_%"] = (by_t.n/by_t.n.sum()*100).round(1)
for c in ["Negatif","Netral","Positif"]: by_t[c+"_%"] = (by_t[c]/by_t.n*100).round(1)
print("\nPER SASARAN:"); print(by_t.sort_values("n", ascending=False)); by_t.sort_values("n", ascending=False).to_csv("hasil/per_sasaran.csv")
by_v = S.groupby(["video_id", "video_title", "channel"]).agg(n=("sent_final", "size"), neg=("sent_final", lambda s: (s == "Negatif").mean()*100)).round(1).reset_index().sort_values("n", ascending=False)
by_v.head(20).to_csv("hasil/per_video_top20.csv", index=False); print("\nTOP VIDEO:"); print(by_v.head(10)[["video_title", "n", "neg"]].to_string(index=False, max_colwidth=60))
pd.concat(SUMMARY).to_csv("hasil/perbandingan_model.csv", index=False)
A.drop(columns=["text_clean"]).to_csv("hasil/prediksi_semua_komentar.csv", index=False, encoding="utf-8-sig")
import json; json.dump({"model_terpilih": CHOICE, "bert_dipakai": (_used["name"] if RUN_BERT else None), "epochs": EPOCHS, "lr": LR, "seed": SEED}, open("hasil/konfigurasi.json", "w"), indent=1)
print("\nFile di folder hasil/:", os.listdir("hasil"))

In [ ]:
# Unduh semua hasil
import shutil; shutil.make_archive("hasil_analisis", "zip", "hasil")
try:
    from google.colab import files; files.download("hasil_analisis.zip")
except Exception: print("hasil_analisis.zip siap")